<!--
Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0
-->


In [ ]:
%store -r

In [ ]:
%store

In [ ]:
# Verify restored variables
import boto3
import os
import pandas as pd
from IPython.display import display

REGION = boto3.Session().region_name

print(f"Region: {REGION}")
print(f"Lambda Function: {LAMBDA_FUNCTION_NAME}")
print(f"Model Package Group: {MODEL_PACKAGE_GROUP_NAME}")
print(f"Ablation Group: {ABLATION_GROUP_NAME}")

# Lab 4: Model Evaluation and SOTA Benchmarking

## Benchmark Against Frontier Models

You now have four Llama 3.2 3B variants scored — base, SFT, SFT+RLVR, and RLVR-only. This lab adds the comparison that puts those numbers in context: **how does a 3B model you spent an hour customizing compare to a frontier model that has never seen your schema?**

Both Claude models are called through Amazon Bedrock with the same schema and question, then scored by the same evaluator Lambda from Lab 2.

### Why these two models

| Model | Role in the comparison |
|---|---|
| **Claude Haiku 4.5** | The fast, low-cost end of the frontier curve. The realistic choice if you were going to solve text-to-SQL by API call at production volume |
| **Claude Sonnet 4.6** | The high-capability end. If raw model strength were the answer, this is where you would see it |

They are two deliberately different points on the **cost/capability curve**, chosen so the result cannot be dismissed as "you picked a weak model". If specialization beats the cheap one but loses to the strong one, that tells you scale substitutes for schema knowledge. If it beats both, that tells you it does not.

::alert[This is not an AWS model recommendation, and it is not a leaderboard result. Claude Sonnet is an extremely capable SQL model — far more capable than Llama 3.2 3B in general. The comparison is narrowly about *schema knowledge on your specific database*, which is the only thing this benchmark measures.]{type="info"}

### What to watch for

The interesting finding usually is not the gap to the fine-tuned models. It is that **Haiku and Sonnet score within about 0.01 of each other.** Roughly an order of magnitude more capability buys essentially nothing here, because the missing ingredient is not reasoning power — it is knowing that this schema calls it `brand_tier`, that "premium products" means a particular value in that column, and which of two plausible date columns the business actually filters on. No amount of scale supplies that; it has to be either prompted in or trained in.

### One methodological caveat, stated plainly

The frontier models and the fine-tuned models reach the scorer by **different paths**, and you should know that before reading the table:

- **Fine-tuned models** were scored by `CustomScorerEvaluator`, which handled prompt construction and inference inside SageMaker.
- **Claude models** are scored by the code below, which hand-rolls its own `SYSTEM_PROMPT` (see the next cell), calls `bedrock.invoke_model` directly, and invokes the evaluator Lambda itself with a `{"batch": [...]}` payload.

The **scorer is identical** — same Lambda, same metrics, same database, same reference queries. What differs is the harness that produces the completions. Note in particular that the code below puts the schema prompt in the `user` role rather than using Bedrock's dedicated `system` field, and uses default sampling parameters. A more carefully tuned prompt would likely lift the Claude scores somewhat.

So treat the frontier rows as a **reasonable, honest baseline rather than a maximally-optimized one**. The gap in this workshop is wide enough that prompt tuning would not reverse the ordering — but it is a real asymmetry and worth naming rather than glossing over.

In [ ]:
import json
import os
import boto3
import time
from botocore.config import Config

# The benchmark makes ~230 sequential invoke_model calls per model. botocore
# retries ThrottlingException on its own, but we set an explicit adaptive
# retry policy so a burst of throttling is absorbed by client-side backoff
# rather than surfacing as empty completions. Adaptive mode adds no delay on
# the happy path -- it only slows down if Bedrock actually throttles us.
bedrock = boto3.client(
    'bedrock-runtime',
    region_name=REGION,
    config=Config(retries={"mode": "adaptive", "max_attempts": 5}),
)
lambda_client = boto3.client('lambda', region_name=REGION)

# Load the validation and combined datasets created in Lab 1
# (01-data-preparation.ipynb, Step 5). If these files are missing, run
# Lab 1 to completion first -- it writes them into the JupyterLab space.
#
# Check first so a skipped/failed Lab 1 gives an actionable message rather than
# a bare FileNotFoundError several lines down.
_missing_files = [f for f in ("validation.jsonl", "combined.jsonl") if not os.path.exists(f)]
if _missing_files:
    raise FileNotFoundError(
        f"Required dataset file(s) not found: {', '.join(_missing_files)}. "
        "These are created in Lab 1 (01-data-preparation.ipynb) and written into "
        "the JupyterLab space. Run Lab 1 to completion first, then re-run this cell."
    )

validation_samples = []
with open('validation.jsonl', 'r') as f:
    for line in f:
        if line.strip():
            validation_samples.append(json.loads(line))

combined_samples = []
with open('combined.jsonl', 'r') as f:
    for line in f:
        if line.strip():
            combined_samples.append(json.loads(line))

print(f"Loaded {len(validation_samples)} validation samples")
print(f"Loaded {len(combined_samples)} combined samples (train + validation)")

# System prompt matching the RLVR/SFT training format
SYSTEM_PROMPT = """You are a SQL query generator for a PostgreSQL database.

DATABASE SCHEMA:

Table: product_sales
  - sale_id (integer NOT NULL)
  - product_name (character varying NOT NULL)
  - product_category (character varying NOT NULL)
  - brand (character varying NOT NULL)
  - brand_tier (character varying NOT NULL)
  - price (numeric NOT NULL)
  - volume_ml (integer)
  - quantity (integer NOT NULL)
  - country_code (character varying NOT NULL)
  - sale_date (date NOT NULL)
  - launch_date (date)
  - customer_segment (character varying NOT NULL)
  - target_age_group (character varying)
  - certification_tags (text)
  - ingredients_type (character varying)
  - season_tag (character varying)
  - rating (numeric)
  - stock_status (character varying NOT NULL)
  - sales_rank (integer)
Generate only the SQL query without explanation."""


def get_bedrock_completion(model_id, user_query):
    response = bedrock.invoke_model(
        modelId=model_id,
        contentType="application/json",
        accept="application/json",
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 256,
            "messages": [
                {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_query}"}
            ],
        }),
    )
    result = json.loads(response['body'].read())
    return result['content'][0]['text'].strip()


def generate_batch(model_id, model_name, samples):
    print(f"  Generating completions for {model_name} ({len(samples)} samples)...")
    batch = []
    for i, sample in enumerate(samples):
        prompt_text = sample['prompt']
        user_query = prompt_text.split("### User Query:\n")[-1].strip()
        try:
            completion = get_bedrock_completion(model_id, user_query)
        except Exception as e:
            print(f"    Sample {i} failed: {e}")
            completion = ""
        batch.append({
            "completion": completion,
            "reference_answer": sample['completion']
        })
        if (i + 1) % 10 == 0:
            print(f"    {i + 1}/{len(samples)} complete")
    return batch


def score_batch(batch, label):
    print(f"  Scoring {label} ({len(batch)} samples)...")
    response = lambda_client.invoke(
        FunctionName=LAMBDA_FUNCTION_NAME,
        InvocationType='RequestResponse',
        Payload=json.dumps({"batch": batch})
    )
    payload = json.loads(response['Payload'].read())
    if 'error' in payload:
        print(f"    Lambda error: {payload['error']}")
        return None
    results = json.loads(payload['body']) if isinstance(payload.get('body'), str) else payload
    metrics = {
        'execution_success': 0.0,
        'execution_accuracy': 0.0,
        'result_set_f1': 0.0,
        'aggregate_reward': 0.0,
    }
    for r in results:
        metrics['aggregate_reward'] += r['aggregate_reward_score']
        for m in r['metrics_list']:
            if m['name'] in metrics:
                metrics[m['name']] += m['value']
    n = len(results)
    for key in metrics:
        metrics[key] = round(metrics[key] / n, 4)
    print(f"    reward={metrics['aggregate_reward']:.3f}, exec={metrics['execution_success']:.3f}, f1={metrics['result_set_f1']:.3f}")
    return metrics


# Evaluate both Claude models against both datasets.
#
# Both use the "us." cross-region inference profile prefix, which is what
# Bedrock expects for these models. The two IDs are not in the same form:
# Haiku carries an explicit version suffix (-20251001-v1:0) while Sonnet uses
# the shorter alias. Both resolve. If you swap in a different model and get
# ValidationException, check the exact profile ID in the Bedrock console
# under Cross-region inference.
MODELS = [
    ("us.anthropic.claude-haiku-4-5-20251001-v1:0", "Claude Haiku 4.5"),
    ("us.anthropic.claude-sonnet-4-6", "Claude Sonnet 4.6"),
]

all_results_val = {}
all_results_combined = {}

for model_id, model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    # Validation set
    val_batch = generate_batch(model_id, model_name, validation_samples)
    val_metrics = score_batch(val_batch, "validation")
    if val_metrics:
        all_results_val[model_name] = val_metrics

    # Combined set
    combined_batch = generate_batch(model_id, model_name, combined_samples)
    combined_metrics = score_batch(combined_batch, "combined")
    if combined_metrics:
        all_results_combined[model_name] = combined_metrics

print("\n\nBenchmark complete!")

## Log Results to MLflow

We log the frontier results to dedicated experiments (`sota-eval-validation` and `sota-eval-combined`) so they sit alongside the fine-tuned model results in the unified comparison without polluting the training experiment namespaces.

Unlike Labs 2 and 3 — where the SageMaker trainers and evaluators logged to MLflow automatically — this notebook calls the `mlflow` client directly, because it ran inference itself rather than through a SageMaker job. Same tracking server, same experiments list, just logged by hand.


In [ ]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_ARN)

# Log SOTA results to the validation experiment
mlflow.set_experiment("sota-eval-validation")
for model_name in all_results_val:
    run_name = model_name.lower().replace(" ", "-").replace(".", "-") + "-baseline"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("evaluation_type", "sota_baseline")
        mlflow.log_param("num_samples", len(validation_samples))
        for key, val in all_results_val[model_name].items():
            mlflow.log_metric(key, val)
        print(f"Logged: {run_name} -> sota-eval-validation")

# Log SOTA combined results to the combined experiment
mlflow.set_experiment("sota-eval-combined")
for model_name in all_results_combined:
    run_name = model_name.lower().replace(" ", "-").replace(".", "-") + "-baseline"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("evaluation_type", "sota_baseline")
        mlflow.log_param("num_samples", len(combined_samples))
        for key, val in all_results_combined[model_name].items():
            mlflow.log_metric(key, val)
        print(f"Logged: {run_name} -> sota-eval-combined")

## Unified Comparison

This is the payoff cell for the whole workshop. It reaches into all eight evaluation experiments — SFT, RLVR, ablation, and the frontier baselines you just logged — and assembles two ranked tables.

Two views, because they answer different questions:

- **Validation set** — held-out queries no model trained on. Measures **generalization**. This is the honest number and the one to quote.
- **Combined set** — every query including the training data. Measures **deployment accuracy** on the real query distribution, since in practice most queries against a schema repeat patterns already seen. Expect these numbers to be higher; that is memorization showing up, and in production it is often a feature rather than a flaw.

### Two mechanics worth understanding, because they are where this cell breaks

**`METRIC_MAP` exists because metric names are not consistent.** The same metric arrives as `aggregate_reward` when this notebook logged it directly, and as `eval/0/aggregate_reward_score` when `CustomScorerEvaluator` logged it — the evaluator Lambda returns the key `aggregate_reward_score`, and different harnesses prefix it differently. The map tries each candidate key in turn. **If a column in your table shows `—`, this is why**: a run logged that metric under a key not in the list.

**`get_display_name()` turns run names into model labels.** SageMaker names its runs `EvaluateBaseModel` and `EvaluateCustomModel`, which alone do not identify which model — so the function combines the run name with the *experiment* name to work out which variant it is looking at. Note that `Llama 3.2 3B (base)` can only come from `sft-eval-*`, because Lab 2 is the only place `evaluate_base_model=True` was set. If that row is missing from your table, the Lab 2 evaluation is the thing to check.

Where the same model appears more than once, the code keeps the highest-scoring row.


In [ ]:
# Collect all evaluation metrics from MLflow
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(MLFLOW_ARN)
client = MlflowClient()


def get_runs_from_experiment(experiment_name):
    exp = client.get_experiment_by_name(experiment_name)
    if not exp:
        print(f"  Experiment '{experiment_name}' not found, skipping")
        return []
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=["start_time DESC"]
    )
    print(f"  {experiment_name}: {len(runs)} run(s)")
    return runs


METRIC_MAP = {
    'execution_success': ['execution_success', 'eval/0/execution_success'],
    'execution_accuracy': ['execution_accuracy', 'eval/0/execution_accuracy'],
    'result_set_f1': ['result_set_f1', 'eval/0/result_set_f1'],
    'aggregate_reward': ['aggregate_reward', 'eval/0/aggregate_reward_score'],
}


def get_display_name(run_name, experiment_name):
    """Map an MLflow run to its comparison-table label.

    Only the Lab 2 SFT evaluators run with evaluate_base_model=True, so
    "Llama 3.2 3B (base)" comes from sft-eval alone. The RLVR and ablation
    evaluators score their custom model only — a base run there would just
    duplicate the Lab 2 row.
    """
    if 'claude' in run_name.lower() or 'haiku' in run_name.lower() or 'sonnet' in run_name.lower():
        return run_name

    is_base = 'EvaluateBaseModel' in run_name or 'BaseModel' in run_name
    is_custom = 'EvaluateCustomModel' in run_name or 'CustomModel' in run_name

    if 'sft-eval' in experiment_name:
        if is_base:
            return "Llama 3.2 3B (base)"
        elif is_custom:
            return "Llama 3.2 3B (SFT)"
    elif 'rlvr-eval' in experiment_name and 'ablation' not in experiment_name:
        if is_custom:
            return "Llama 3.2 3B (SFT + RLVR)"
    elif 'ablation' in experiment_name:
        if is_custom:
            return "Llama 3.2 3B (RLVR only)"

    return run_name


def extract_metrics(runs, experiment_name=""):
    results = []
    for run in runs:
        run_name = run.info.run_name or run.info.run_id[:8]
        metrics = run.data.metrics
        display_name = get_display_name(run_name, experiment_name)
        row = {"Model": display_name}

        for display_key, possible_keys in METRIC_MAP.items():
            for key in possible_keys:
                if key in metrics:
                    row[display_key] = metrics[key]
                    break

        if len(row) > 1:
            results.append(row)

    return results


# Collect VALIDATION results
print("Collecting VALIDATION results...")
val_results = []
for exp_name in ["sft-eval-validation", "rlvr-eval-validation", "rlvr-ablation-eval-validation", "sota-eval-validation"]:
    runs = get_runs_from_experiment(exp_name)
    val_results.extend(extract_metrics(runs, experiment_name=exp_name))

for model_name, metrics in all_results_val.items():
    if not any(r['Model'] == model_name for r in val_results):
        val_results.append({"Model": model_name, **metrics})

seen = {}
for row in val_results:
    name = row['Model']
    if name not in seen or row.get('aggregate_reward', 0) > seen[name].get('aggregate_reward', 0):
        seen[name] = row
val_results = list(seen.values())

# Collect COMBINED results
print("\nCollecting COMBINED results...")
combined_results = []
for exp_name in ["sft-eval-combined", "rlvr-eval-combined", "rlvr-ablation-eval-combined", "sota-eval-combined"]:
    runs = get_runs_from_experiment(exp_name)
    combined_results.extend(extract_metrics(runs, experiment_name=exp_name))

for model_name, metrics in all_results_combined.items():
    if not any(r['Model'] == model_name for r in combined_results):
        combined_results.append({"Model": model_name, **metrics})

seen = {}
for row in combined_results:
    name = row['Model']
    if name not in seen or row.get('aggregate_reward', 0) > seen[name].get('aggregate_reward', 0):
        seen[name] = row
combined_results = list(seen.values())

print(f"\nDone: {len(val_results)} validation results, {len(combined_results)} combined results")

In [ ]:
# Display validation set results
from IPython.display import display, HTML
display(HTML("<h3>HELD-OUT VALIDATION SET (unseen queries — measures generalization)</h3>"))
def format_table(results):
    df = pd.DataFrame(results)
    if 'aggregate_reward' in df.columns:
        df = df.sort_values('aggregate_reward', ascending=False)
    display_df = df.copy()
    for col in ['execution_success', 'execution_accuracy', 'result_set_f1', 'aggregate_reward']:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "—")
    display_df = display_df.rename(columns={
        'execution_success': 'Exec Success',
        'execution_accuracy': 'Exact Match',
        'result_set_f1': 'Result F1',
        'aggregate_reward': 'Reward Score',
    })
    return display_df.set_index('Model')


display(format_table(val_results))

In [ ]:
# Display combined production set results
display(HTML("<h3>COMBINED SET (all queries, train + validation — measures deployment accuracy)</h3>"))
display(format_table(combined_results))

## Key Finding

Read your two tables together. The conclusions below should hold in your run even though the exact numbers will not match any reference run.

1. **SFT + RLVR ranks first, with plain SFT close behind.** The RLVR increment is genuinely small on ~200 examples — if those two swap places in your table, that is noise on a 37-sample split, not a finding.
2. **Both fine-tuned models clearly beat RLVR-only and the base model**, which confirms Lab 3's ablation from a second direction.
3. **The fine-tuned 3B beats both frontier models by a wide margin** on Exact Match and Result F1. This gap is large enough that it should not invert.
4. **Exec Success is high for everyone (~0.95+).** Every model here writes valid SQL. All of the separation comes from Exact Match and Result F1 — whether the query returns the *right answer* against your schema.
5. **Haiku and Sonnet land within ~0.01 of each other.** An order of magnitude of capability difference buys essentially nothing on this task.

### What this does and does not claim

Be precise about the conclusion, because it is easy to overstate.

**Claude Sonnet is not a worse SQL model than Llama 3.2 3B.** It is a far better one in general — better reasoning, better at unfamiliar schemas, better at complex multi-step queries, and it needs no training at all. What it does not have is any knowledge of *your* tables, *your* column names, *your* join conventions, or the business meaning your team attaches to a value like `brand_tier = 'premium'`. No amount of scale supplies that.

So the finding is narrow and real: **on a fixed, known schema, a few hundred real examples buy you more than several orders of magnitude of model size.** That is a statement about where the difficulty in text-to-SQL actually lives.

It is also worth stating the reverse honestly. If your schema changed weekly, if you needed the model to handle databases it had never seen, or if you had no query history to learn from, the frontier model would be the better engineering choice — and you would reach for prompt engineering and RAG rather than fine-tuning. The decision ladder in the workshop introduction is the thing to take away, not this single table.

### Remember the harness asymmetry

As noted at the top of the notebook: the frontier rows came from a hand-rolled Bedrock call with a fixed prompt in the `user` role, while the fine-tuned rows came through `CustomScorerEvaluator`. Same scorer, different prompt-construction path. A tuned prompt would likely lift the Claude numbers some — not enough to reverse the ordering here, but the rows are a fair baseline rather than a maximum.

One more thing to watch, since it cuts the *other* way. The Bedrock loop retries throttling automatically (adaptive backoff, up to five attempts), but if a run hits *sustained* throttling that exhausts those retries, the affected sample's completion comes back empty and scores zero — which drags down the **frontier** model's average, not the fine-tuned one. So if a Claude row looks suspiciously low and you saw `Sample N failed` lines during generation, that is a throttling artifact, not a capability result. Re-run the cell rather than reading it as a finding.


## Review Everything in MLflow

Your tracking server now holds all eleven experiments from the workshop. The cell below links straight to it.

The **Reviewing Results** module that follows walks through what to look for in each one — the SFT loss curve, the two RLVR reward curves side by side, and how to read the evaluation runs.


In [ ]:
# Open MLflow. Every experiment the workshop produces is listed below, so the
# count is derived from the lists rather than written out as a number that can
# drift out of step with them.
from IPython.display import display, HTML

_sm = boto3.client("sagemaker", region_name=REGION)
try:
    _url = _sm.create_presigned_mlflow_app_url(Arn=MLFLOW_ARN)["AuthorizedUrl"]
    _label = "Open MLflow (signed link, valid ~5 minutes)"
except Exception as e:
    print(f"Presigned URL unavailable ({type(e).__name__}), falling back to console link.")
    _url = f"https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/mlflow"
    _label = "Open MLflow in the AWS Console"

display(HTML(f'<a href="{_url}" target="_blank"><b>{_label}</b></a>'))

TRAINING_EXPERIMENTS = ("sft-training", "rlvr-training", "rlvr-base-training")
EVALUATION_EXPERIMENTS = (
    "sft-eval-validation", "sft-eval-combined",
    "rlvr-eval-validation", "rlvr-eval-combined",
    "rlvr-ablation-eval-validation", "rlvr-ablation-eval-combined",
    "sota-eval-validation", "sota-eval-combined",
)

print(f"\n{len(TRAINING_EXPERIMENTS) + len(EVALUATION_EXPERIMENTS)} experiments from the workshop:")
print(f"\nTraining ({len(TRAINING_EXPERIMENTS)}):")
for _e in TRAINING_EXPERIMENTS:
    print(f"  - {_e}")
print(f"\nEvaluation ({len(EVALUATION_EXPERIMENTS)}):")
for _e in EVALUATION_EXPERIMENTS:
    # The two sota-eval-* experiments were written by this notebook directly;
    # the other six came from SageMaker evaluators in Labs 2 and 3.
    print(f"  - {_e}")
